<a href="https://colab.research.google.com/github/Siddesh-2004/FindYourPhone/blob/main/notebook/featureBasedLaptopRecom.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [47]:
# ═══════════════════════════════════════════════════════════════
# CELL 1 — Imports
# ═══════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

In [48]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — Load & Clean
# ═══════════════════════════════════════════════════════════════
df = pd.read_csv("laptopPrice_train.csv")

# Drop duplicates
df.drop_duplicates(inplace=True)

# Drop low-signal columns (review counts, ms office flag, os_bit)
dead_cols = ["Number of Ratings", "Number of Reviews", "msoffice", "os_bit"]
df.drop(columns=dead_cols, inplace=True)

# ── Extract numeric fields ──────────────────────────────────────
df["ram_gb"]           = df["ram_gb"].str.extract(r"(\d+)").astype(float).astype("Int64")
df["ssd_gb"]           = df["ssd"].str.extract(r"(\d+)").astype(float).astype("Int64")
df["hdd_gb"]           = df["hdd"].str.extract(r"(\d+)").astype(float).astype("Int64")
df["graphic_card_gb"]  = df["graphic_card_gb"].str.extract(r"(\d+)").astype(float).astype("Int64")
df["warranty_yrs"]     = df["warranty"].str.extract(r"(\d+)").astype(float).fillna(0).astype("Int64")
df["rating_num"]       = df["rating"].str.extract(r"(\d+)").astype(float)

df.drop(columns=["ssd", "hdd", "warranty", "rating"], inplace=True)

# ── Boolean / binary flags ──────────────────────────────────────
df["Touchscreen"] = df["Touchscreen"].map(lambda x: 1 if str(x).strip().lower() == "yes" else 0).astype("Int64")

# ── Processor generation → ordinal ─────────────────────────────
gen_map = {"7th": 7, "8th": 8, "9th": 9, "10th": 10, "11th": 11,
           "12th": 12, "13th": 13, "Not Available": 0}
df["processor_gnrtn"] = df["processor_gnrtn"].map(gen_map).fillna(0).astype("Int64")

# ── Weight category → ordinal (Casual < ThinNlight < Gaming) ───
weight_map = {"Casual": 1, "ThinNlight": 2, "Gaming": 3}
df["weight_cat"] = df["weight"].map(weight_map).fillna(1).astype("Int64")
df.drop(columns=["weight"], inplace=True)

print("After cleaning:", df.shape)
df.head(3)

After cleaning: (644, 15)


,brand,processor_brand,processor_name,processor_gnrtn,ram_gb,ram_type,os,graphic_card_gb,Touchscreen,Price,ssd_gb,hdd_gb,warranty_yrs,rating_num,weight_cat
0,ASUS,Intel,Core i5,11,16,DDR4,Windows,0,0,65990,512,0,1,3.0,1
1,acer,AMD,Ryzen 5,0,4,DDR4,Windows,0,0,44990,512,0,1,4.0,1
2,ASUS,AMD,Ryzen 7,0,4,DDR4,Windows,4,0,72400,512,0,0,3.0,1


In [49]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 — Outlier Capping (IQR) & Low-Variance Drop
# ═══════════════════════════════════════════════════════════════
features = ['ram_gb', 'ssd_gb', 'hdd_gb', 'graphic_card_gb',
            'processor_gnrtn', 'rating_num', 'Price']

for col in features:
    df[col] = df[col].astype(float)
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    df[col] = df[col].clip(lower=q1 - 1.5 * iqr, upper=q3 + 1.5 * iqr)

print("After outlier capping:", df.shape)

After outlier capping: (644, 15)


In [50]:
# ═══════════════════════════════════════════════════════════════
# CELL 4 — Normalize Features
# ═══════════════════════════════════════════════════════════════
df_model = df.dropna(subset=features).reset_index(drop=True)

scaler = MinMaxScaler()
df_model[features] = scaler.fit_transform(df_model[features])

print("After normalization:", df_model.shape)

After normalization: (644, 15)


In [51]:
# ═══════════════════════════════════════════════════════════════
# CELL 5 — Rule-Based Filter
# ═══════════════════════════════════════════════════════════════
def rule_based_filter(df, budget=None, os=None, processor_brand=None,
                      need_touchscreen=False, weight_type=None):
    """
    Hard filters applied before cosine scoring.

    Parameters
    ----------
    budget           : int   — max price (INR)
    os               : str   — 'Windows' | 'Mac' | 'DOS'
    processor_brand  : str   — 'Intel' | 'AMD' | 'M1'
    need_touchscreen : bool  — filter to touchscreen laptops only
    weight_type      : str   — 'Casual' | 'ThinNlight' | 'Gaming'
    """
    filtered = df.copy()
    if budget is not None:
        filtered = filtered[filtered["Price"] <= budget]
    if os is not None:
        filtered = filtered[filtered["os"] == os]
    if processor_brand is not None:
        filtered = filtered[filtered["processor_brand"] == processor_brand]
    if need_touchscreen:
        filtered = filtered[filtered["Touchscreen"] == 1]
    if weight_type is not None:
        wmap = {"Casual": 1, "ThinNlight": 2, "Gaming": 3}
        filtered = filtered[filtered["weight_cat"] == wmap.get(weight_type, 1)]
    return filtered.reset_index(drop=True)

In [52]:
# ═══════════════════════════════════════════════════════════════
# CELL 6 — Cosine Similarity Scorer and Euclidean Distance
# ═══════════════════════════════════════════════════════════════
from sklearn.neighbors import NearestNeighbors

def map_weights(feature_importance: dict) -> np.ndarray:
    """
    Converts user slider values (1–5) to weight multipliers (1.0–3.0).
    Formula: weight = 1.0 + (slider - 1) * 0.5
    Slider 1 → 1.0, Slider 3 → 2.0, Slider 5 → 3.0
    """
    return np.array([
        1.0 + (feature_importance.get(f, 3) - 1) * 0.5
        for f in features
    ])


def build_user_vector(ram_gb=8, ssd_gb=512, hdd_gb=0, graphic_card_gb=0,
                      processor_gnrtn=11, rating_num=4.0, Price=50000):
    """
    Build and scale a user preference vector using the same scaler
    fitted on the training data.
    """
    raw = pd.DataFrame(
        [[ram_gb, ssd_gb, hdd_gb, graphic_card_gb, processor_gnrtn, rating_num, Price]],
        columns=features
    )
    return scaler.transform(raw)


def euclidean_similarity(user_vector, laptop_matrix, weights):
    """
    Computes normalized Euclidean similarity between user vector and all laptops.

    Steps:
      1. Apply weights to both sides
      2. Compute row-wise Euclidean distance
      3. Normalize by theoretical max distance in weighted [0,1] space
         → max possible distance = sqrt(sum of weights²)
      4. Flip to similarity: 1 - normalized_distance

    Returns array of shape (n,) with values in [0, 1].
    """
    weighted_user   = user_vector * weights        # shape (1, 7)
    weighted_matrix = laptop_matrix * weights      # shape (n, 7)

    diff      = weighted_matrix - weighted_user    # broadcasting
    distances = np.linalg.norm(diff, axis=1)

    max_distance = np.sqrt(np.sum(weights ** 2))

    return 1 - (distances / max_distance)


def score_laptops(candidate_df, user_vector, weights):
    """
    Hybrid score = 0.6 × cosine_similarity + 0.4 × euclidean_similarity
    Both terms are in [0, 1], higher = better match.
    """
    weighted_user   = user_vector * weights
    weighted_matrix = candidate_df[features].values * weights

    cosine_scores    = cosine_similarity(weighted_user, weighted_matrix)[0]
    euclidean_scores = euclidean_similarity(user_vector, candidate_df[features].values, weights)

    hybrid_scores = 0.6 * cosine_scores + 0.4 * euclidean_scores

    result = candidate_df.copy()
    result["match_score"] = (hybrid_scores * 100).round(2)
    return result.sort_values("match_score", ascending=False).reset_index(drop=True)


def score_laptops_knn(candidate_df, user_vector, weights, top_n=10):
    """
    KNN-based scorer. Retrieves nearest neighbors in the weighted feature space.
    Returns a df with match_score column (0–100), same shape as score_laptops output.
    """
    X = candidate_df[features].values.astype(float)

    X_weighted  = X * weights
    user_vec    = (user_vector * weights).reshape(1, -1)

    k     = min(top_n, len(X_weighted))
    index = NearestNeighbors(n_neighbors=k, metric="euclidean")
    index.fit(X_weighted)

    distances, indices = index.kneighbors(user_vec, n_neighbors=k)
    dist = distances[0]

    max_possible = np.sqrt(np.sum(weights ** 2))
    scores = np.clip(100 * (1 - dist / max_possible), 0, 100).round(2)

    result = candidate_df.iloc[indices[0]].copy()
    result["match_score"] = scores
    return result.sort_values("match_score", ascending=False).reset_index(drop=True)

In [53]:
# ═══════════════════════════════════════════════════════════════
# CELL 7 — recommend() function
# ═══════════════════════════════════════════════════════════════

DEFAULT_IMPORTANCE = {
    "ram_gb": 3, "ssd_gb": 3, "hdd_gb": 3,
    "graphic_card_gb": 3, "processor_gnrtn": 3,
    "rating_num": 3, "Price": 3
}

def recommend(budget=None, os=None, processor_brand=None,
              need_touchscreen=False, weight_type=None,
              ram_gb=8, ssd_gb=512, hdd_gb=0, graphic_card_gb=0,
              processor_gnrtn=11, rating_num=4.0, Price=50000,
              feature_importance=None, top_n=10, knn_pool=50):

    # Fall back to neutral defaults if no importance provided
    importance = feature_importance if feature_importance is not None else DEFAULT_IMPORTANCE

    candidates = rule_based_filter(
        df_model,
        budget=budget, os=os,
        processor_brand=processor_brand,
        need_touchscreen=need_touchscreen,
        weight_type=weight_type
    )

    if candidates.empty:
        print("No laptops match your filters. Try relaxing some constraints.")
        return pd.DataFrame()

    user_vector = build_user_vector(
        ram_gb=ram_gb, ssd_gb=ssd_gb, hdd_gb=hdd_gb,
        graphic_card_gb=graphic_card_gb,
        processor_gnrtn=processor_gnrtn,
        rating_num=rating_num,
        Price=Price
    )

    weights = map_weights(importance)

    # Stage 1 — KNN narrows the pool to closest neighbors
    knn_pool_size = min(knn_pool, len(candidates))
    knn_shortlist = score_laptops_knn(candidates, user_vector, weights, top_n=knn_pool_size)

    # Stage 2 — Hybrid cosine+euclidean re-scores the shortlist
    ranked = score_laptops(knn_shortlist, user_vector, weights)

    display_cols = ["brand", "processor_brand", "processor_name",
                    "os", "Price", "match_score"]
    display_cols = [c for c in display_cols if c in ranked.columns]
    return ranked[display_cols].head(top_n)

In [54]:
# ═══════════════════════════════════════════════════════════════
# CELL 8 — 3 Persona Demos
# ═══════════════════════════════════════════════════════════════
personas = [
    {
        "name": "Student",
        "params": dict(
            budget=35000, os="Windows", processor_brand="Intel",
            need_touchscreen=False, weight_type="ThinNlight",
            ram_gb=8, ssd_gb=256, hdd_gb=0, graphic_card_gb=0,
            processor_gnrtn=10, rating_num=3.5, Price=30000
        )
    },
    {
        "name": "Content Creator",
        "params": dict(
            budget=100000, os="Windows", processor_brand="Intel",
            need_touchscreen=True, weight_type=None,
            ram_gb=16, ssd_gb=512, hdd_gb=0, graphic_card_gb=4,
            processor_gnrtn=12, rating_num=4.5, Price=85000
        )
    },
    {
        "name": "Gamer",
        "params": dict(
            budget=120000, os="Windows", processor_brand="Intel",
            need_touchscreen=False, weight_type="Gaming",
            ram_gb=16, ssd_gb=512, hdd_gb=1024, graphic_card_gb=8,
            processor_gnrtn=12, rating_num=4.5, Price=100000
        )
    },
]

for p in personas:
    print(f"\n{'='*60}")
    print(f"  Persona: {p['name']}")
    print(f"{'='*60}")
    result = recommend(**p["params"])
    if not result.empty:
        print(result.to_string(index=False))


  Persona: Student
 brand processor_brand processor_name      os    Price  match_score
  ASUS           Intel        Core i3 Windows 0.123564        97.47
Lenovo           Intel        Core i3 Windows 0.152727        97.30
Lenovo           Intel        Core i3 Windows 0.152727        97.30
  ASUS           Intel        Core i3 Windows 0.145455        97.18
  ASUS           Intel        Core i3 Windows 0.152727        97.14
Lenovo           Intel        Core i3 Windows 0.160000        97.09
Lenovo           Intel        Core i3 Windows 0.167273        97.03
    HP           Intel        Core i3 Windows 0.167273        97.03
Lenovo           Intel        Core i3 Windows 0.181818        96.90
  DELL           Intel        Core i3 Windows 0.181818        96.90

  Persona: Content Creator
 brand processor_brand processor_name      os    Price  match_score
   MSI           Intel        Core i7 Windows 1.000000        86.12
  ASUS           Intel        Core i7 Windows 1.000000        85.27


In [55]:
# ═══════════════════════════════════════════════════════════════
# CELL 9 — Golden Set Evaluation  (fixed)
# ═══════════════════════════════════════════════════════════════

def run_golden_set():
    passed = 0
    failed = 0

    def check(label, condition):
        nonlocal passed, failed
        if condition:
            print(f"  [ PASS ] {label}")
            passed += 1
        else:
            print(f"  [ FAIL ] {label}")
            failed += 1

    # ── helper: returns full rows ────────────────────────────────
    def recommend_full(budget=None, os=None, processor_brand=None,
                       need_touchscreen=False, weight_type=None,
                       ram_gb=8, ssd_gb=512, hdd_gb=0, graphic_card_gb=0,
                       processor_gnrtn=11, rating_num=4.0, Price=50000,
                       top_n=10):
        candidates = rule_based_filter(
            df_model,
            budget=budget, os=os,
            processor_brand=processor_brand,
            need_touchscreen=need_touchscreen,
            weight_type=weight_type
        )
        if candidates.empty:
            return pd.DataFrame()
        user_vector = build_user_vector(
            ram_gb=ram_gb, ssd_gb=ssd_gb, hdd_gb=hdd_gb,
            graphic_card_gb=graphic_card_gb,
            processor_gnrtn=processor_gnrtn,
            rating_num=rating_num,
            Price=Price
        )
        weights = map_weights(DEFAULT_IMPORTANCE)
        knn_shortlist = score_laptops_knn(candidates, user_vector, weights, top_n=min(50, len(candidates)))
        ranked = score_laptops(knn_shortlist, user_vector, weights)
        return ranked.head(top_n)

    # ── pre-compute the normalized budget thresholds ─────────────
    price_min = scaler.data_min_[features.index("Price")]
    price_max = scaler.data_max_[features.index("Price")]
    def norm_price(inr):
        return (inr - price_min) / (price_max - price_min)

    # dataset-level averages (on the normalized scale)
    avg_ram     = df_model["ram_gb"].mean()
    avg_ssd     = df_model["ssd_gb"].mean()
    avg_graphic = df_model["graphic_card_gb"].mean()

    def score_floor(results, percentile=75):
        """Return the `percentile`-th score in a full candidate run."""
        return np.percentile(results["match_score"].values, percentile)

    # ───────────────────────────────────────────────
    # TEST GROUP 1 — Student
    # ───────────────────────────────────────────────
    print("\n=== Student ===")
    student = recommend_full(
        budget=norm_price(35000), os="Windows", processor_brand="Intel",
        need_touchscreen=False, weight_type="ThinNlight",
        ram_gb=8, ssd_gb=256, hdd_gb=0, graphic_card_gb=0,
        processor_gnrtn=10, rating_num=3.5, Price=30000
    )

    check("Student — returned at least 1 result", not student.empty)
    check("Student — all top 5 run Windows", (student.head(5)["os"] == "Windows").all())
    check("Student — all top 5 are Intel", (student.head(5)["processor_brand"] == "Intel").all())
    check("Student — all top 5 are ThinNlight (weight_cat == 2)", (student.head(5)["weight_cat"] == 2).all())
    check("Student — top 5 normalized Price ≤ norm(₹35,000)", (student.head(5)["Price"] <= norm_price(35000) + 1e-6).all())
    check("Student — top 1 match score is above the 75th percentile of candidates", student.iloc[0]["match_score"] >= score_floor(student, 75))

    # ───────────────────────────────────────────────
    # TEST GROUP 2 — Content Creator
    # ───────────────────────────────────────────────
    print("\n=== Content Creator ===")
    creator = recommend_full(
        budget=norm_price(100000), os="Windows", processor_brand="Intel",
        need_touchscreen=True, weight_type=None,
        ram_gb=16, ssd_gb=512, hdd_gb=0, graphic_card_gb=4,
        processor_gnrtn=12, rating_num=4.5, Price=85000
    )

    check("Creator — returned at least 1 result", not creator.empty)
    check("Creator — all top 5 have Touchscreen", (creator.head(5)["Touchscreen"] == 1).all())
    check("Creator — top 5 normalized Price ≤ norm(₹1,00,000)", (creator.head(5)["Price"] <= norm_price(100000) + 1e-6).all())
    check("Creator — top 5 avg RAM above dataset average", creator.head(5)["ram_gb"].mean() > avg_ram)
    check("Creator — top 5 avg SSD above dataset average", creator.head(5)["ssd_gb"].mean() > avg_ssd)
    check("Creator — top 5 avg graphic card above dataset average", creator.head(5)["graphic_card_gb"].mean() > avg_graphic)
    check("Creator — top 1 match score is above the 75th percentile of candidates", creator.iloc[0]["match_score"] >= score_floor(creator, 75))

    # ───────────────────────────────────────────────
    # TEST GROUP 3 — Gamer
    # ───────────────────────────────────────────────
    print("\n=== Gamer ===")
    gamer = recommend_full(
        budget=norm_price(120000), os="Windows", processor_brand="Intel",
        need_touchscreen=False, weight_type="Gaming",
        ram_gb=16, ssd_gb=512, hdd_gb=1024, graphic_card_gb=8,
        processor_gnrtn=12, rating_num=4.5, Price=100000
    )

    check("Gamer — returned at least 1 result", not gamer.empty)
    check("Gamer — all top 5 are Gaming (weight_cat == 3)", (gamer.head(5)["weight_cat"] == 3).all())
    check("Gamer — top 5 normalized Price ≤ norm(₹1,20,000)", (gamer.head(5)["Price"] <= norm_price(120000) + 1e-6).all())
    check("Gamer — top 5 avg RAM above dataset average", gamer.head(5)["ram_gb"].mean() > avg_ram)
    check("Gamer — top 5 avg graphic card above dataset average", gamer.head(5)["graphic_card_gb"].mean() > avg_graphic)
    check("Gamer — top 1 is ranked higher than top 5 median score", gamer.iloc[0]["match_score"] >= gamer.head(5)["match_score"].median())

    # ───────────────────────────────────────────────
    # TEST GROUP 4 — Edge Cases
    # ───────────────────────────────────────────────
    print("\n=== Edge Cases ===")

    impossible = recommend_full(budget=norm_price(100), os="Windows")
    check("Impossible budget (₹100) returns empty results", impossible.empty)

    impossible_combo = recommend_full(budget=norm_price(200000), os="Mac", processor_brand="AMD")
    check("Conflicting filters (Mac + AMD) returns empty or handles gracefully",
          impossible_combo.empty or len(impossible_combo) >= 0)

    try:
        high_budget = recommend_full(
            budget=norm_price(500000), os="Windows",
            ram_gb=64, ssd_gb=2000, graphic_card_gb=16,
            processor_gnrtn=13, rating_num=5.0, Price=400000
        )
        check("Extreme specs (max everything) returns results without crash", not high_budget.empty)
    except Exception as e:
        check(f"Extreme specs returns results without crash — ERROR: {e}", False)

    # ───────────────────────────────────────────────
    # TEST GROUP 5 — Score Sanity
    # ───────────────────────────────────────────────
    print("\n=== Score Sanity ===")

    base = dict(
        budget=norm_price(120000), os="Windows", processor_brand="Intel",
        weight_type="Gaming", ram_gb=16, ssd_gb=512, hdd_gb=1024,
        processor_gnrtn=12, rating_num=4.5, Price=100000
    )

    high_gpu = recommend_full(**base, graphic_card_gb=8, top_n=1)
    low_gpu  = recommend_full(**base, graphic_card_gb=0, top_n=50)

    top_brand  = high_gpu.iloc[0]["brand"]
    high_score = high_gpu.iloc[0]["match_score"]
    low_row    = low_gpu[low_gpu["brand"] == top_brand]
    low_score  = low_row.iloc[0]["match_score"] if not low_row.empty else 0

    check("Top gaming laptop scores higher with graphic_card_gb=8 vs 0", high_score >= low_score)
    check("All match scores are in [0, 100]", high_gpu["match_score"].between(0, 100).all())
    check("Results are sorted descending by match_score", gamer["match_score"].is_monotonic_decreasing)

    # ───────────────────────────────────────────────
    # TEST GROUP 6 — Filter Correctness
    # ───────────────────────────────────────────────
    print("\n=== Filter Correctness ===")

    ts_results = recommend_full(budget=norm_price(80000), need_touchscreen=True, top_n=10)
    check("need_touchscreen=True — all top 10 have Touchscreen", (ts_results["Touchscreen"] == 1).all())

    amd_results = recommend_full(budget=norm_price(80000), processor_brand="AMD", top_n=10)
    check("processor_brand='AMD' — all top 10 are AMD", (amd_results["processor_brand"] == "AMD").all())

    casual_results = recommend_full(budget=norm_price(50000), weight_type="Casual", top_n=10)
    check("weight_type='Casual' — all top 10 are Casual (weight_cat == 1)", (casual_results["weight_cat"] == 1).all())

    # ───────────────────────────────────────────────
    # TEST GROUP 7 — Cross-Persona Comparison
    # ───────────────────────────────────────────────
    print("\n=== Cross-Persona Comparison ===")

    student_cross = recommend_full(
        budget=norm_price(35000), os="Windows", weight_type="ThinNlight",
        ram_gb=8, ssd_gb=256, graphic_card_gb=0,
        processor_gnrtn=10, rating_num=3.5, Price=30000, top_n=5
    )
    gamer_cross = recommend_full(
        budget=norm_price(120000), os="Windows", weight_type="Gaming",
        ram_gb=16, ssd_gb=512, hdd_gb=1024, graphic_card_gb=8,
        processor_gnrtn=12, rating_num=4.5, Price=100000, top_n=5
    )

    check("Gamer top 5 has higher avg graphic_card_gb than Student top 5",
          gamer_cross["graphic_card_gb"].mean() > student_cross["graphic_card_gb"].mean())
    check("Gamer top 5 has higher avg RAM than Student top 5",
          gamer_cross["ram_gb"].mean() > student_cross["ram_gb"].mean())
    check("Student top 5 has lower avg Price than Gamer top 5",
          student_cross["Price"].mean() < gamer_cross["Price"].mean())

    # ───────────────────────────────────────────────
    # SUMMARY
    # ───────────────────────────────────────────────
    total = passed + failed
    print(f"\n{'═'*50}")
    print(f"  Results: {passed}/{total} passed", end="  ")
    print("✓ All good!" if failed == 0 else f"✗ {failed} assertion(s) failed — check above")
    print(f"{'═'*50}")

run_golden_set()


=== Student ===
  [ PASS ] Student — returned at least 1 result
  [ PASS ] Student — all top 5 run Windows
  [ PASS ] Student — all top 5 are Intel
  [ PASS ] Student — all top 5 are ThinNlight (weight_cat == 2)
  [ PASS ] Student — top 5 normalized Price ≤ norm(₹35,000)
  [ PASS ] Student — top 1 match score is above the 75th percentile of candidates

=== Content Creator ===
  [ PASS ] Creator — returned at least 1 result
  [ PASS ] Creator — all top 5 have Touchscreen
  [ PASS ] Creator — top 5 normalized Price ≤ norm(₹1,00,000)
  [ PASS ] Creator — top 5 avg RAM above dataset average
  [ PASS ] Creator — top 5 avg SSD above dataset average
  [ PASS ] Creator — top 5 avg graphic card above dataset average
  [ PASS ] Creator — top 1 match score is above the 75th percentile of candidates

=== Gamer ===
  [ PASS ] Gamer — returned at least 1 result
  [ PASS ] Gamer — all top 5 are Gaming (weight_cat == 3)
  [ PASS ] Gamer — top 5 normalized Price ≤ norm(₹1,20,000)
  [ PASS ] Gamer — to